# SkyFare Predictor — Flight Fare Prediction

End-to-end pipeline: data cleaning -> feature engineering -> model comparison -> cross-validation -> feature importance -> model export.

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import warnings
warnings.filterwarnings("ignore")

df = pd.read_excel("Flight Fare/Data_Train.xlsx")
print(df.shape)
df.head()

## 2. Data Cleaning

EDA revealed: 220 duplicate rows, 1 row with missing `Route`/`Total_Stops`, inconsistent casing in `Additional_Info` (`'No Info'` vs `'No info'`), and one corrupted row (a 2-stop flight with `Duration` listed as `'5m'`, which is physically impossible).

In [ ]:
print("Nulls:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows")

df = df.dropna(subset=["Total_Stops", "Route"])

df["Additional_Info"] = df["Additional_Info"].str.strip().replace({"No Info": "No info"})

df = df[df["Duration"] != "5m"]
print("Shape after cleaning:", df.shape)

## 3. Feature Engineering

**Important fix vs the original version of this project:** duration is parsed directly from the `Duration` string column reported by the airline (e.g. `'2h 50m'`), not derived by subtracting `Arrival_hour - Dep_hour`. The subtraction approach silently breaks for any flight that crosses midnight (departs 23:30, arrives 02:15 -> naive subtraction gives a nonsensical value). Parsing the actual duration string avoids this entirely, both here and in the Flask app at prediction time.

In [ ]:
df["Date_of_Journey"] = pd.to_datetime(df["Date_of_Journey"], format="%d/%m/%Y")
df["Journey_day"] = df["Date_of_Journey"].dt.day
df["Journey_month"] = df["Date_of_Journey"].dt.month
df["Journey_weekday"] = df["Date_of_Journey"].dt.weekday

df["Dep_hour"] = pd.to_datetime(df["Dep_Time"]).dt.hour
df["Dep_min"] = pd.to_datetime(df["Dep_Time"]).dt.minute

df["Arrival_hour"] = df["Arrival_Time"].apply(lambda x: int(x.split(" ")[0].split(":")[0]))
df["Arrival_min"] = df["Arrival_Time"].apply(lambda x: int(x.split(" ")[0].split(":")[1]))

def parse_duration(dur_str):
    hours = re.search(r"(\d+)h", dur_str)
    mins = re.search(r"(\d+)m", dur_str)
    return (int(hours.group(1)) if hours else 0, int(mins.group(1)) if mins else 0)

durations = df["Duration"].apply(parse_duration)
df["Duration_hours"] = durations.apply(lambda x: x[0])
df["Duration_mins"] = durations.apply(lambda x: x[1])
df["Duration_total_mins"] = df["Duration_hours"] * 60 + df["Duration_mins"]

df = df[df["Duration_total_mins"] > 0]

stops_map = {"non-stop": 0, "1 stop": 1, "2 stops": 2, "3 stops": 3, "4 stops": 4}
df["Total_Stops"] = df["Total_Stops"].map(stops_map)

df_model = df.drop(columns=["Date_of_Journey", "Dep_Time", "Arrival_Time",
                             "Duration", "Route", "Additional_Info"])
df_model.head()

**Note on `Total_Stops` encoding:** encoded as an ordinal integer (0,1,2,3,4) rather than one-hot, because stop count has a genuine natural order. One-hot would throw away that ordering information.

**Note on dropping `Route`:** dropped because it's largely redundant with `Source` + `Destination` + `Total_Stops` combined, and one-hot encoding it would add dozens of very sparse, high-cardinality columns prone to overfitting on a dataset this size.

## 4. Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = df.groupby("Airline")["Price"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="Airline", y="Price", order=order, ax=axes[0])
axes[0].set_title("Price Distribution by Airline")
axes[0].tick_params(axis='x', rotation=75)

sns.boxplot(data=df, x="Total_Stops", y="Price", ax=axes[1])
axes[1].set_title("Price by Number of Stops")

plt.tight_layout()
plt.show()

Jet Airways Business fares sit far above every other class (expected - business class). Price also rises with number of stops, which makes sense: multi-stop itineraries are often longer-haul or off-peak routings.

## 5. Train/Test Split + Preprocessing Pipeline

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X = df_model.drop(columns=["Price"])
y = df_model["Price"]

categorical_cols = ["Airline", "Source", "Destination"]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)],
    remainder="passthrough"
)

Using a `ColumnTransformer` + `Pipeline` here instead of manually one-hot encoding with if/elif chains (as the original app.py did). This guarantees training-time and inference-time encoding can never drift apart, and the entire pipeline (preprocessing + model) is saved as ONE object.

## 6. Model Comparison

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results.append({"Model": name, "RMSE": round(rmse, 2), "MAE": round(mae, 2), "R2": round(r2, 4)})
    fitted_pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df

## 7. Cross-Validation on Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipe = fitted_pipelines[best_model_name]
print("Best model:", best_model_name)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_pipe, X, y, cv=cv, scoring="neg_root_mean_squared_error")
cv_rmse = -cv_scores

print("Fold RMSEs:", np.round(cv_rmse, 2))
print(f"Mean RMSE: {cv_rmse.mean():.2f}  |  Std: {cv_rmse.std():.2f}")

## 8. Feature Importance

In [ ]:
ohe = best_pipe.named_steps["preprocessor"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
all_feature_names = list(cat_feature_names) + numeric_cols

importances = best_pipe.named_steps["model"].feature_importances_
fi_df = pd.DataFrame({"feature": all_feature_names, "importance": importances}).sort_values("importance", ascending=False)
fi_df.head(10)

## 9. Save Final Pipeline

In [ ]:
with open("flight_price_pipeline.pkl", "wb") as f:
    pickle.dump(best_pipe, f)

print("Saved flight_price_pipeline.pkl")